# Trusted Zone

## Initialization

In [1]:
import os
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, length, trim, when, udf
from pyspark.sql.types import StringType
import datetime
from bs4 import BeautifulSoup
from langdetect import detect

In [4]:
jdbc_jar_path = "../formatted_zone/postgresql-42.7.3.jar"
if 'spark' not in locals():
    if not os.path.exists(jdbc_jar_path):
         print(f"Error: JDBC Jar not found at {jdbc_jar_path}")
         # Handle error appropriately
    else:
        spark = SparkSession.builder \
            .appName("TrustedZoneETL") \
            .config("spark.jars", jdbc_jar_path) \
            .getOrCreate()
    print("Spark Session (re)started for Trusted Zone.")
else:
    print("Using existing Spark Session for Trusted Zone.")

Using existing Spark Session for Trusted Zone.


In [5]:
pg_url = "jdbc:postgresql://localhost:5432/bdaproject2"
pg_properties = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver"
}

if isinstance(pg_properties["password"], bytes):
    pg_properties["password"] = pg_properties["password"].decode("utf-8", errors="replace")

## Movies Database (TMDB)

In [ ]:
print("\nProcessing Movies Table (Trusted Zone)...")
try:
    formatted_movies_df = spark.read \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "movies") \
        .load()

    initial_count = formatted_movies_df.count()
    print(f"Read {initial_count} rows from formatted movies table.")

    # Data Quality que hem fet:
    # Completeness
    # Consistency: Budget, revenue, runtime, and vote counts cannot be negative.
    # Range Check: vote_average between 0 and 10.
    # Temporal Consistency: Release dates should not be too far in the future.
    # We assume that revenue = 0 and budget means missing values
    
    trusted_movies_df = formatted_movies_df.filter(
        col("movie_id").isNotNull() & \
        col("title").isNotNull() & (trim(col("title")) != "") & \
        col("release_date").isNotNull() & \
        (col("budget") > 0) & \
        (col("revenue") > 0) & \
        (col("runtime") >= 0) & \
        (col("vote_count") >= 0) & \
        (col("vote_average").between(0, 10)) & \
        (col("release_date") < datetime.date(2030, 1, 1))
    )

    trusted_movies_df.cache()
    final_count = trusted_movies_df.count()
    print(f"Applying DQ rules removed {initial_count - final_count} rows.")
    print(f"Writing {final_count} trusted rows back to movies table.")

    trusted_movies_df.write \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "movies") \
        .mode("overwrite") \
        .save()

    print("Movies table quality checked and written to PostgreSQL (Trusted).")
    trusted_movies_df.show(5, truncate=50)

except Exception as e:
    print(f"Error processing Movies table in Trusted Zone: {e}")


Processing Movies Table (Trusted Zone)...
Read 1514 rows from formatted movies table.
Applying DQ rules removed 170 rows.
Writing 1344 trusted rows back to movies table.
Movies table quality checked and written to PostgreSQL (Trusted).
+--------+-----------------------------------+------------------------------------+--------------------------------------------------+------------+--------+---------+-------+----------+------------+----------+--------+----------------------------------------+-----+--------------------------------+---------+-----------------+-------------+---------------------------------------+
|movie_id|                              title|                      original_title|                                          overview|release_date|  budget|  revenue|runtime|popularity|vote_average|vote_count|  status|                                 tagline|adult|                     poster_path|  imdb_id|original_language|collection_id|                        collection_name|
+

In [15]:
print("\nProcessing Movie Genres Table (Trusted Zone)...")
try:
    formatted_genres_df = spark.read \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "movie_genres") \
        .load()

    initial_count = formatted_genres_df.count()
    print(f"Read {initial_count} rows from formatted movie_genres table.")

    trusted_genres_df = formatted_genres_df.filter(
        col("movie_id").isNotNull() & \
        col("genre_id").isNotNull() & \
        col("genre_name").isNotNull() & (trim(col("genre_name")) != "")
    )

    trusted_genres_df.cache()
    final_count = trusted_genres_df.count()
    print(f"Applying DQ rules removed {initial_count - final_count} rows.")
    print(f"Writing {final_count} trusted rows back to movie_genres table.")

    trusted_genres_df.write \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "movie_genres") \
        .mode("overwrite") \
        .save()

    print("Movie Genres table quality checked and written to PostgreSQL (Trusted).")
    trusted_genres_df.show(5)

except Exception as e:
    print(f"Error processing Movie Genres table in Trusted Zone: {e}")


Processing Movie Genres Table (Trusted Zone)...
Read 6637 rows from formatted movie_genres table.
Applying DQ rules removed 0 rows.
Writing 6637 trusted rows back to movie_genres table.
Movie Genres table quality checked and written to PostgreSQL (Trusted).
+--------+--------+----------+
|movie_id|genre_id|genre_name|
+--------+--------+----------+
|    1359|      53|  Thriller|
|    1359|      18|     Drama|
|    1359|      80|     Crime|
|    1599|      35|    Comedy|
|    1599|   10749|   Romance|
+--------+--------+----------+
only showing top 5 rows



In [16]:
print("\nProcessing Movie Production Companies Table (Trusted Zone)...")
try:
    formatted_companies_df = spark.read \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "movie_production_companies") \
        .load()

    initial_count = formatted_companies_df.count()
    print(f"Read {initial_count} rows from formatted movie_production_companies table.")

    # Data Quality Rules (Referential Integrity & Completeness):
    trusted_companies_df = formatted_companies_df.filter(
        col("movie_id").isNotNull() & \
        col("company_id").isNotNull() & \
        col("company_name").isNotNull() & (trim(col("company_name")) != "")
    )

    trusted_companies_df.cache()
    final_count = trusted_companies_df.count()
    print(f"Applying DQ rules removed {initial_count - final_count} rows.")
    print(f"Writing {final_count} trusted rows back to movie_production_companies table.")

    trusted_companies_df.write \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "movie_production_companies") \
        .mode("overwrite") \
        .save()

    print("Movie Production Companies table quality checked and written to PostgreSQL (Trusted).")
    trusted_companies_df.show(5, truncate=50)

except Exception as e:
    print(f"Error processing Movie Production Companies table in Trusted Zone: {e}")


Processing Movie Production Companies Table (Trusted Zone)...
Read 7235 rows from formatted movie_production_companies table.
Applying DQ rules removed 0 rows.
Writing 7235 trusted rows back to movie_production_companies table.
Movie Production Companies table quality checked and written to PostgreSQL (Trusted).
+--------+----------+---------------------+--------------+
|movie_id|company_id|         company_name|origin_country|
+--------+----------+---------------------+--------------+
|     156|       223| Les Films du Losange|            FR|
|     156|       235| Nordisk Film Denmark|            DK|
|     156|       698|      Scottish Screen|            GB|
|     156|       758|                 TV 2|            DK|
|     156|       980|The Glasgow Film Fund|            GB|
+--------+----------+---------------------+--------------+
only showing top 5 rows



In [ ]:
print("\nProcessing Movie Cast Table (Trusted Zone)...")
try:
    formatted_cast_df = spark.read \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "movie_cast") \
        .load()

    initial_count = formatted_cast_df.count()
    print(f"Read {initial_count} rows from formatted movie_cast table.")

    trusted_cast_df = formatted_cast_df.filter(
        col("credit_id").isNotNull() & (trim(col("credit_id")) != "") & \
        col("movie_id").isNotNull() & \
        col("actor_id").isNotNull() & \
        col("actor_name").isNotNull() & (trim(col("actor_name")) != "")
    )

    trusted_cast_df.cache()
    final_count = trusted_cast_df.count()
    print(f"Applying DQ rules removed {initial_count - final_count} rows.")
    print(f"Writing {final_count} trusted rows back to movie_cast table.")

    trusted_cast_df.write \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "movie_cast") \
        .mode("overwrite") \
        .save()

    print("Movie Cast table quality checked and written to PostgreSQL (Trusted).")
    trusted_cast_df.show(5, truncate=50)

except Exception as e:
    print(f"Error processing Movie Cast table in Trusted Zone: {e}")


Processing Movie Cast Table (Trusted Zone)...
Read 95548 rows from formatted movie_cast table.
Applying DQ rules removed 0 rows.
Writing 95548 trusted rows back to movie_cast table.
Movie Cast table quality checked and written to PostgreSQL (Trusted).
+------------------------+--------+--------+----------------+-------------------+----------+------+--------------------------------+
|               credit_id|movie_id|actor_id|      actor_name|     character_name|cast_order|gender|                    profile_path|
+------------------------+--------+--------+----------------+-------------------+----------+------+--------------------------------+
|52fe423cc3a36847f800e6c1|     378|    2963|    Nicolas Cage|               H.I.|         0|     2|/ar33qcWbEgREn07ZpXv5Pbj8hbM.jpg|
|52fe423cc3a36847f800e6c5|     378|   18686|    Holly Hunter|                 Ed|         1|     1|/kC7KX03VAWvogOCuwKbMo4V6TuU.jpg|
|52fe423cc3a36847f800e6c9|     378|    4039|     Trey Wilson|Nathan Arizona, Sr.| 

In [12]:
print("\nProcessing Movie Crew Table (Trusted Zone)...")
try:
    formatted_crew_df = spark.read \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "movie_crew") \
        .load()

    initial_count = formatted_crew_df.count()
    print(f"Read {initial_count} rows from formatted movie_crew table.")

    # Data Quality Rules (Primary Key, Foreign Key & Completeness):
    # credit_id, job, and department are essential crew information.
    trusted_crew_df = formatted_crew_df.filter(
        col("credit_id").isNotNull() & (trim(col("credit_id")) != "") & \
        col("movie_id").isNotNull() & \
        col("crew_member_id").isNotNull() & \
        col("crew_member_name").isNotNull() & (trim(col("crew_member_name")) != "") & \
        col("department").isNotNull() & (trim(col("department")) != "") & \
        col("crew_job").isNotNull() & (trim(col("crew_job")) != "")
    )

    trusted_crew_df.cache()
    final_count = trusted_crew_df.count()
    print(f"Applying DQ rules removed {initial_count - final_count} rows.")
    print(f"Writing {final_count} trusted rows back to movie_crew table.")

    trusted_crew_df.write \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "movie_crew") \
        .mode("overwrite") \
        .save()

    print("Movie Crew table quality checked and written to PostgreSQL (Trusted).")
    trusted_crew_df.show(5, truncate=50)

except Exception as e:
    print(f"Error processing Movie Crew table in Trusted Zone: {e}")


Processing Movie Crew Table (Trusted Zone)...
Read 152996 rows from formatted movie_crew table.
Applying DQ rules removed 0 rows.
Writing 152996 trusted rows back to movie_crew table.
Movie Crew table quality checked and written to PostgreSQL (Trusted).
+------------------------+--------+--------------+-----------------+----------+-----------------------+------+--------------------------------+
|               credit_id|movie_id|crew_member_id| crew_member_name|department|               crew_job|gender|                    profile_path|
+------------------------+--------+--------------+-----------------+----------+-----------------------+------+--------------------------------+
|52fe4259c3a36847f8017823|     596|          8507|    Mark-Lee Kirk|       Art|          Art Direction|     2|                            NULL|
|52fe4259c3a36847f8017817|     596|          8505|Robert L. Simpson|   Editing|                 Editor|     2|                            NULL|
|52fe4259c3a36847f80177ff

## OMDB database
Movie reviews

In [18]:
print("\nProcessing OMDb Movie Details Table (Trusted Zone)...")
try:
    formatted_omdb_df = spark.read \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "omdb_movie_details") \
        .load()

    initial_count = formatted_omdb_df.count()
    print(f"Read {initial_count} rows from formatted omdb_movie_details table.")

    trusted_omdb_df = formatted_omdb_df.filter(
        col("imdb_id").isNotNull() & (trim(col("imdb_id")) != "") & \
        col("title").isNotNull() & (trim(col("title")) != "") & \
        col("released").isNotNull() & \
        
        # Validity
        col("imdb_id").startswith("tt") & \
        (col("type") == 'movie') & \
        # There was only, one series -> it has different columns... To easy
        # our work, we prefer to assume we only have movies. 
        
        (col("year").isNotNull() & col("year").between(1888, 2030)) & \
        (col("metascore").isNull() | col("metascore").between(0, 100)) & \
        (col("imdb_rating").isNull() | col("imdb_rating").between(0.0, 10.0)) & \
        (col("imdb_votes").isNull() | (col("imdb_votes") >= 0)) & \

        (col("released") < datetime.date(2030, 1, 1))
    )

    trusted_omdb_df.cache()
    final_count = trusted_omdb_df.count()
    print(f"Applying DQ rules removed {initial_count - final_count} rows.")
    print(f"Writing {final_count} trusted rows back to omdb_movie_details table.")
    
    trusted_omdb_df.write \
        .format("jdbc") \
        .options(**pg_properties) \
        .option("url", pg_url) \
        .option("dbtable", "omdb_movie_details") \
        .mode("overwrite") \
        .save()

    print("OMDb Movie Details table quality checked and written to PostgreSQL (Trusted).")
    trusted_omdb_df.show(5, truncate=50)

except Exception as e:
    print(f"Error processing OMDb Movie Details table in Trusted Zone: {e}")
    import traceback
    traceback.print_exc()


Processing OMDb Movie Details Table (Trusted Zone)...
Read 1000 rows from formatted omdb_movie_details table.
Applying DQ rules removed 2 rows.
Writing 998 trusted rows back to omdb_movie_details table.
OMDb Movie Details table quality checked and written to PostgreSQL (Trusted).
+---------+-------------------------------------+----+--------+----------+-------+--------------------------+-----------------+--------------------------------------------------+--------------------------------------------------+--------------------------------------------------+------------------------+-----------------------------------------+--------------------------------------------------+--------------------------------------------------+---------+-----------+----------+-----+--------+------------+----------+-------+--------------------------------------------------+
|  imdb_id|                                title|year|   rated|  released|runtime|                     genre|         director|          